1. Define Dictionaries and Parameters


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

from ipywidgets import Dropdown, Output, IntText, ToggleButton, Text
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
import json
import os
import subprocess
import numpy as np
import time

# --- Define your dictionaries ---
xy = [949, 1169]  # Example, adjust as needed

d13 = {
    'scantype': 'map',
    'roicenter': xy,
    'prefix': 'eiger4m_',
    'suffix': 'h5',
    'folder': '/data/visitor/blc16859/bm32/20260210/RAW_DATA/NiTi_Heraud/NiTi_Heraud_map/scan0013',
    'CCDLabel': 'EIGER_4MCdTe',
    'listindices': list(range(41 * (2482 // 41))),
    'mapdimensions': (41, 201),
    'nbimagesperline': 41,
    'fastaxis': 'yech',
    'slowaxis': 'xech'
}

d5 = {
    'scantype': 'map',
    'roicenter': xy,
    'prefix': 'eiger4m_',
    'suffix': 'h5',
    'folder': '/data/visitor/blc16859/bm32/20260210/RAW_DATA/NiTi_Heraud/NiTi_Heraud_NiTi2/scan0005',
    'CCDLabel': 'EIGER_4MCdTe',
    'listindices': list(range(21 * 41)),
    'mapdimensions': (21, 41),
    'nbimagesperline': 21,
    'fastaxis': 'yech',
    'slowaxis': 'xech'
}

# --- Default parameters ---
boxsize_X, boxsize_Y = 13, 13
AUTOCONTRAST = False
ACQUISITON_ON = False
nbcompletelines = 3  #  None

# --- Widgets for parameters ---
boxsize_X_widget = IntText(value=boxsize_X, description='boxsize_X:')
boxsize_Y_widget = IntText(value=boxsize_Y, description='boxsize_Y:')
AUTOCONTRAST_widget = ToggleButton(value=AUTOCONTRAST, description='AUTOCONTRAST', disabled=False)
ACQUISITON_ON_widget = ToggleButton(value=ACQUISITON_ON, description='ACQUISITON_ON', disabled=False)
nbcompletelines_widget = Text(value=str(nbcompletelines), description='nbcompletelines:', disabled=False)

# --- Display parameter widgets ---
display(boxsize_X_widget, boxsize_Y_widget, AUTOCONTRAST_widget, ACQUISITON_ON_widget, nbcompletelines_widget)

In [ ]:
# --- Define available partitions and their configurations ---
partition_configs = {
    'magnifix': {'nbcpus': 96, 'partition': 'magnifix', 'constraint': 'NONE', 'comment': '#', 'nodes': 1},
    'magnifix2': {'nbcpus': 96 * 2, 'partition': 'magnifix', 'constraint': 'NONE', 'comment': '#', 'nodes': 1},
    'ub20': {'nbcpus': 96, 'partition': 'ub20', 'constraint': 'NONE', 'comment': '#', 'nodes': 1},
    'hpc7': {'nbcpus': 96, 'partition': 'nice', 'constraint': 'hpc7', 'comment': '', 'nodes': 1},
    'hpc8': {'nbcpus': 64, 'partition': 'nice', 'constraint': 'hpc8', 'comment': '', 'nodes': 1},
    'hpc6': {'nbcpus': 40, 'partition': 'nice', 'constraint': 'hpc6', 'comment': '', 'nodes': 1},
    'lbm32gpu1': {'nbcpus': 72, 'partition': None, 'constraint': 'NONE', 'comment': '#', 'nodes': 1},
    'hpc6-01': {'nbcpus': 64, 'partition': None, 'constraint': 'NONE', 'comment': '#', 'nodes': 1},
    'hpc6-02': {'nbcpus': 64, 'partition': None, 'constraint': 'NONE', 'comment': '#', 'nodes': 1},
    'hpc6-03': {'nbcpus': 64, 'partition': None, 'constraint': 'NONE', 'comment': '#', 'nodes': 1},
    'hpc6-04': {'nbcpus': 64, 'partition': None, 'constraint': 'NONE', 'comment': '#', 'nodes': 1}
}

# --- Create dropdown widget for partition selection ---
partition_dropdown = Dropdown(
    options=list(partition_configs.keys()),
    value='ub20',  # Default to 'ub20'
    description='Partition:',
    disabled=False,
)

# --- Create dropdown widget for dictionary selection ---
dict_dropdown = Dropdown(
    options=['d13', 'd5'],
    value='d13',
    description='Dictionary:',
    disabled=False,
)

# --- Display widgets ---
display(partition_dropdown, dict_dropdown)



In [ ]:
# --- Function to update partition settings ---
def update_partition_config(change):
    global ESRFmachine, nbcpus, partition, constraint, comment, nodes, forced_ncpus
    selected_partition = change['new']
    config = partition_configs[selected_partition]
    ESRFmachine = selected_partition
    nbcpus = config['nbcpus']
    partition = config['partition']
    constraint = config['constraint']
    comment = config['comment']
    nodes = config['nodes']
    forced_ncpus = max(32, nbcpus)

# --- Function to update selected dictionary ---
def update_selected_dict(change):
    global selected_dict
    selected_dict = globals()[change['new']]

# --- Attach the update functions to the dropdowns ---
partition_dropdown.observe(update_partition_config, names='value')
dict_dropdown.observe(update_selected_dict, names='value')

# --- Initialize with default values ---
update_partition_config({'new': partition_dropdown.value})
update_selected_dict({'new': dict_dropdown.value})

print('ESRFmachine',ESRFmachine)
print('selected_dict',selected_dict)

In [ ]:
script_folder = "/data/bm32/inhouse/LAUESCRIPTS/notebooks/LaueTools_script/"
script_python_file = "mosaic_plot.py"
script_path = str(Path(script_folder+'//'+script_python_file))

# --- Generate config file ---
config = {
    "d": selected_dict,
    "boxsize_X": boxsize_X_widget.value,
    "boxsize_Y": boxsize_Y_widget.value,
    "AUTOCONTRAST": AUTOCONTRAST_widget.value,
    "ACQUISITON_ON": ACQUISITON_ON_widget.value,
    "nbcompletelines": (int(nbcompletelines_widget.value) if nbcompletelines_widget.value not in ("", "None") else None
)
}

# Convert tuples to lists for JSON serialization
for key, value in config["d"].items():
    if isinstance(value, tuple):
        config["d"][key] = list(value)

config_file_path = os.path.join(selected_dict['folder'], 'mosaic_config.json')
with open(config_file_path, 'w') as f:
    json.dump(config, f, indent=4)

print(f"Config file generated at: {config_file_path}")

# --- Generate SLURM script ---
folder_slurm_file = Path(script_folder)  # Replace with your path
slurm_file = str(folder_slurm_file / f'job_indexing_{ESRFmachine}.slurm')
job_name = f"mosaic_{Path(selected_dict['folder']).name}"

# Skip SLURM submission if partition is None (e.g., lbm32gpu1 or hpc6-01/02/03/04)
if partition is None:
    print(f"Partition '{ESRFmachine}' does not use SLURM. Running locally...")
    # Run the script directly (adjust as needed)
    !python {script_path} --config {config_file_path} --ncpus {forced_ncpus}
else:
    slurm_script = f"""#!/bin/bash
                    #SBATCH --job-name="{job_name}"
                    #SBATCH --mail-type=NONE
                    #SBATCH --mail-user=micha@esrf.fr
                    #SBATCH --partition={partition}
                    #SBATCH --nodes={nodes}
                    #SBATCH --ntasks={forced_ncpus}
                    #SBATCH --mem=200GB
                    #SBATCH --output={selected_dict['folder']}/%x_out.txt
                    #SBATCH --error={selected_dict['folder']}/%x_err.txt
                    {comment}#SBATCH --constraint="{constraint}"
                    
                    echo "Date              = $(date)"
                    echo "Hostname          = $(hostname -s)"
                    echo "Working Directory = $(pwd)"
                    
                    # Load environment
                    module load mamba
                    conda activate /data/bm32/inhouse/SOFT/lauetoolsDEV
                    
                    # Run the script
                    python {script_path} --config {config_file_path} --ncpus {forced_ncpus}
                    """

    with open(slurm_file, 'w') as f:
        f.write(slurm_script)

    print(f"SLURM script generated at: {slurm_file}")

    

In [ ]:
# Submit the job **ONCE** and capture the job ID
    result = subprocess.run(['sbatch', slurm_file], capture_output=True, text=True)
    job_id = result.stdout.strip().split()[-1]  # Extract job ID from stdout
    print(f"Submitted job ID: {job_id}")

3. Poll for Job Completion and Plot Results


In [ ]:
results_file

In [ ]:
# --- Poll for job completion (optional) ---
# --- Poll for job completion (optional) ---
if 1:
    if partition is not None:
        print(f"Waiting for SLURM job {job_id} to finish...")
        while True:
            status = subprocess.getoutput(f"squeue -j {job_id}")
            if not status:
                print("Job finished!")
                break
            time.sleep(10)  # Check every 10 seconds

# --- Load results and plot ---
results_file = os.path.join(selected_dict['folder'], 'allresults.npy')
if os.path.exists(results_file):
    allresults = np.load(results_file)
    print(f"Loaded results with shape: {allresults.shape}")
else:
    print(f"Results file not found at {results_file}. Did the job finish?")
    # If running locally, results may be in memory or saved elsewhere
    # Adjust as needed for your workflow


# Reuse your plotting logic here
dimfast, dimslow = selected_dict['mapdimensions']
mosaic = np.zeros((dimslow, dimfast, 2 * boxsize_Y_widget.value + 1, 2 * boxsize_X_widget.value + 1))
sm = mosaic.shape
bigimage = np.zeros((sm[0] * sm[2], sm[1] * sm[3]))

if dimfast > 0:
    for map_imageindex, absolute_imageindex in enumerate(selected_dict['listindices']):
        imap, jmap = map_imageindex // dimfast, map_imageindex % dimfast
        raw = allresults[map_imageindex, :, :]
        datcrop = raw
        mosaic[imap, jmap] = datcrop
        bigimage[imap * sm[2]:(imap + 1) * sm[2], jmap * sm[3]:(jmap + 1) * sm[3]] = np.flipud(datcrop)

mosaictranspose = mosaic.transpose((0, 3, 1, 2))
mosaicflat = mosaictranspose.reshape((dimfast * (2 * boxsize_X_widget.value + 1), dimslow * (2 * boxsize_Y_widget.value + 1)))

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
vmin, vmax = (None, None) if AUTOCONTRAST_widget.value else (3.2, 3.5)
ax.imshow(np.log10(bigimage + 0.001), origin='lower', vmin=vmin, vmax=vmax)

if selected_dict['fastaxis'] == 'yech':
    ax.set_xlabel(f"fastaxis {selected_dict['fastaxis']} // pixelY")
    ax.set_ylabel(f"slowaxis {selected_dict['slowaxis']} // pixelX")
else:
    ax.set_xlabel(f"fastaxis {selected_dict['fastaxis']} // pixelX")
    ax.set_ylabel(f"slowaxis {selected_dict['slowaxis']} // pixelY")

ax.set_title(f"{selected_dict['folder']} roicenter X,Y = ({selected_dict['roicenter'][0]},{selected_dict['roicenter'][1]})")
plt.show()

3. Submit SLURM Job and Wait for Completion
